# 21 · WGFMU IV sweep · L1 空连 (real B1500, no DUT)

> ⚠️ **执行前必读** ⚠️
>
> 此 notebook 会真的连 B1500，但**严禁接器件**。目的是验证：
> 1. `wgfmu.dll` 能加载
> 2. VISA 通路打通
> 3. `RealWgfmuBackend` 整条 IV sweep 链路通
> 4. 开路条件下电流读数接近 0
>
> 必须在执行前确认：
> - [ ] B1500 已开机、自检通过
> - [ ] Keysight IO Libraries 装好，Connection Expert 能看到 B1500
> - [ ] **WGFMU CH101 探针抬起或断开，没有任何 DUT 接入**
> - [ ] 项目 venv 装好 (`pip install -r requirements/dev.txt` + `pip install -e .`)

**通过标准**：
- 无 dll 错
- `complete == total`
- `qc.status == "ok"`
- 开路电流 |I_mean| < 1 µA (远低于量程 1 mA)


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Python:", sys.version.split()[0])
print("project root:", ROOT)


## 1. VISA 自检 (确认 B1500 地址 + 在线)


In [ ]:
import pyvisa

rm = pyvisa.ResourceManager()
resources = rm.list_resources()
print("Available VISA resources:")
for r in resources:
    print(" -", r)

# 默认地址 (来自 src/scripts/connect_test.py)
VISA_ADDR = "GPIB0::17::INSTR"

if VISA_ADDR not in resources:
    print()
    print(f"⚠️  默认地址 {VISA_ADDR} 不在可用列表里")
    print("    请从上面 list 里选一个 B1500，改下面这一行后重跑这个 cell")
else:
    print(f"\n✅ {VISA_ADDR} 可用")


In [ ]:
# 拿到 *IDN? 确认是 B1500
inst = rm.open_resource(VISA_ADDR)
inst.write_termination = "\r\n"
inst.read_termination = "\r\n"
inst.timeout = 10000

try:
    idn = inst.query("*IDN?").strip()
    print("*IDN? ->", idn)
    assert "B1500" in idn or "Agilent" in idn or "Keysight" in idn, \
        f"unexpected instrument: {idn}"
    print("✅ B1500 confirmed")
finally:
    inst.close()
rm.close()


## 2. 加载 wgfmu.dll


In [ ]:
from fefetlab.measurements.wgfmu.real_backend import RealWgfmuBackend

backend = RealWgfmuBackend()
try:
    backend.load()
    print("✅ wgfmu.dll loaded")
except OSError as e:
    print("❌ DLL load failed:")
    print(e)
    print()
    print("常见排查:")
    print(" 1. wgfmu.dll 是否在 PATH 或 C:\\Program Files\\Keysight\\B1530A\\bin")
    print(" 2. 是否 32/64 位 Python 与 dll 不匹配")
    raise


## 3. open_session + 通道枚举


In [ ]:
backend.open_session(VISA_ADDR)
backend.set_timeout(60.0)

channel_ids = backend.get_channel_ids()
print(f"detected WGFMU channels: {channel_ids}")

CHAN_ID = 101
if CHAN_ID not in channel_ids:
    print(f"⚠️  CHAN_ID={CHAN_ID} 不在检测到的通道里")
    print(f"    从 {channel_ids} 里挑一个改这里")
else:
    print(f"✅ CH{CHAN_ID} 可用")


## 4. 最小 IV sweep · 极低压·小量程·开路验证

参数选择理由：
- `v_stop = -0.5V`：远低于任何 FeFET 阈值，开路即便误碰也安全
- `n_points = 5`：最少必要
- `current_range = "1UA"`：高灵敏度，开路应该 < 100 nA
- `t_high = 5e-6`：足够采到 20 个点


In [ ]:
from fefetlab.measurements.wgfmu.pulse_builder import linear_voltage_segments
from fefetlab.measurements.wgfmu.iv_sweep import WgfmuIVSweepRunner, WgfmuIVSweepConfig

segments = linear_voltage_segments(
    v_start=0.0,
    v_stop=-0.5,
    n_points=5,
    t_rise_s=1e-6,
    t_high_s=5e-6,
    t_fall_s=1e-6,
    t_base_s=2e-6,
    measure_points=20,
    measure_average_s=100e-9,
)

cfg = WgfmuIVSweepConfig(
    label="L1_aircheck_ivsweep",
    chan_id=CHAN_ID,
    v_init=0.0,
    v_base=0.0,
    operation_mode="FASTIV",
    force_voltage_range="AUTO",
    measure_mode="CURRENT",
    measure_current_range="1UA",   # 高灵敏度
    treat_warning_as_error=False,
    timeout_s=30.0,
    notes="L1 空连验证, 探针抬起, 期望 |I| < 100 nA",
)

print("segments:", len(segments), "expected pulses: 5")
print("cfg:", cfg.label)


In [ ]:
# === 真正调用 B1500 ===
# 注意: 这里会启动硬件，确保探针抬起!
runner = WgfmuIVSweepRunner(backend=backend)

result = runner.run(
    resource=VISA_ADDR,
    segments=segments,
    cfg=cfg,
)

print("✅ run() returned")
print(f"complete / total : {result.complete} / {result.total}")
print(f"issues           : {result.issues}")
print(f"channels detected: {result.channel_ids}")


## 5. QC + 数据检视


In [ ]:
print("=== qc_df ===")
print(result.qc_df.to_string(index=False))
print()
print("=== iv_df (per-pulse summary) ===")
print(result.iv_df.to_string(index=False))
print()
print("=== samples (head 10) ===")
print(result.samples_df.head(10).to_string(index=False))
print(f"\ntotal samples: {len(result.samples_df)}")


In [ ]:
# 错误/警告 dump
err = result.meta.get("error_summary", "")
wrn = result.meta.get("warning_summary", "")
print("error_summary:")
print(err if err else "  (empty)")
print()
print("warning_summary:")
print(wrn if wrn else "  (empty)")
print()
print(f"execution_time_s: {result.meta.get('execution_time_s'):.2f}")


## 6. 画图: time-current (开路应近 0)


In [ ]:
samples = result.samples_df
value_col = "current_A" if "current_A" in samples.columns else samples.columns[-1]

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

# upper: voltage waveform reconstructed
t_wf, v_wf = result.plan.waveform_samples(dt_s=5e-8)
axes[0].plot(t_wf*1e6, v_wf, color="#1f77b4", lw=1.0)
axes[0].set_ylabel("voltage (V)")
axes[0].set_title(f"L1 aircheck · IV sweep · {result.iv_df.shape[0]} pulses")
axes[0].grid(alpha=0.3)

# lower: measured current
axes[1].plot(samples["time_s"]*1e6, samples[value_col]*1e9, "o-",
             ms=3, lw=0.8, color="#d62728")
axes[1].set_xlabel("time (us)")
axes[1].set_ylabel(f"{value_col} (nA)")
axes[1].grid(alpha=0.3)
axes[1].axhline(0, color="gray", lw=0.5)

plt.tight_layout()
plt.show()

# 数字判据
i_abs_max = samples[value_col].abs().max()
i_abs_mean = samples[value_col].abs().mean()
print(f"\n|I|_max  = {i_abs_max*1e9:.3f} nA")
print(f"|I|_mean = {i_abs_mean*1e9:.3f} nA")

# 1 µA 阈值: 开路下应远低于
if i_abs_max < 1e-6:
    print("✅ 开路验证通过 (|I|_max < 1 uA)")
else:
    print(f"⚠️  |I|_max = {i_abs_max*1e6:.3f} uA >= 1 uA")
    print("    可能原因: 1) 探针未真正抬起 2) 误接 DUT 3) 量程不当 4) 接地干扰")


## 7. 关闭硬件 + 落盘路径打印


In [ ]:
print("output paths:")
for k, v in result.paths.items():
    print(f"  {k:20s} -> {v}")

print(f"\nrun directory: {Path(list(result.paths.values())[0]).parent}")


---

## 通过标准回顾

- [ ] cell 1: `*IDN?` 返回含 "B1500"
- [ ] cell 2: dll 加载无错
- [ ] cell 3: `get_channel_ids()` 含 101
- [ ] cell 4-5: `complete == total`，无 issues
- [ ] cell 6: `|I|_max < 1 uA` (开路条件)
- [ ] cell 7: output 路径打印正常

**全部 ✅ → 进 23 (wake-up L1 空连)**

**任何异常 / warning / |I| 偏大都停下来发给阿耶，不要硬上器件**
